# Notebook 01 — Spatial Health Data
**GeoAI for Climate, Environment and Health** | 25 June 2026

---

## Learning objectives
By the end of this notebook you will be able to:
- Load and inspect a geocoded health dataset
- Understand coordinate reference systems and what latitude/longitude mean
- Produce a basic spatial scatter map with matplotlib
- Identify a common data-type bug that silently corrupts ID fields

**Time:** ~25 minutes

---
### How this notebook works
| Symbol | Meaning |
|--------|--------|
| 📖 | Read the concept note |
| ✅ | Worked example — run the cell and observe |
| 🧪 | Your turn — fill in the `...` and run |
| ✔ | Auto-check — tells you if your answer is correct |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import sys, pathlib

def _find_workshop_dir():
    here = pathlib.Path().resolve()
    for candidate in (here, here.parent, here.parent.parent):
        if (candidate / 'data').exists():
            return candidate
    return here

WORKSHOP_DIR = _find_workshop_dir()  # works whether run flat or from a notebooks/ subfolder
DATA_DIR = WORKSHOP_DIR / 'data'

print('Ready.')

---
## 📖 1. What is geocoded health data?

**Geocoded** data means every record has a location attached — typically a latitude and longitude. In health research this is powerful: it lets us ask *where* disease burden is concentrated, *how far* people travel for care, and *what environmental conditions* surround a patient or facility.

Common geocoded health datasets you will encounter:

| Dataset type | What each row represents | Typical location info |
|---|---|---|
| Health facility register | A clinic, hospital, or health post | GPS coordinates of the building |
| Demographic & Health Survey (DHS) | A household or individual | Cluster centroid (GPS-displaced for privacy) |
| Surveillance linelist | A case of a disease | Patient's village or grid cell |
| Pregnancy cohort (e.g. PRECISE) | A pregnant woman | Village reported at each study visit |

**Coordinate Reference Systems (CRS)**

Most field GPS devices and web services use **WGS 84** (EPSG:4326) — the same system used by Google Maps and smartphones. In this CRS:
- **Latitude** runs north–south: negative = south of the equator (Zimbabwe is ~15°S–22°S)
- **Longitude** runs east–west: positive = east of the Greenwich meridian (Zimbabwe is ~26°E–33°E)

> **Rule of thumb for Zimbabwe:** latitude is a negative number between -15 and -22; longitude is a positive number between 26 and 33. Any coordinate outside these ranges is almost certainly wrong.

---
## ✅ Worked example 1 — Load and inspect health facilities

In [ ]:
# Load the dataset.
facilities = pd.read_csv(DATA_DIR / 'health_facilities.csv')

print(f'Shape: {facilities.shape}  ({facilities.shape[0]} facilities, {facilities.shape[1]} columns)\n')
print('Column dtypes:')
print(facilities.dtypes)
print()
facilities.head()

In [ ]:
# Check for missing coordinates — records without a location cannot be mapped.
missing_coords = facilities[facilities['latitude'].isna() | facilities['longitude'].isna()]
print(f'Facilities with missing coordinates: {len(missing_coords)}')
print(missing_coords[['facility_id', 'facility_type', 'district']])

---
## 🧪 Exercise 1 — Count facilities per district  *(2 points)*

Count how many facilities are in each district and sort the result from most to fewest.

Store your answer in a variable called `facilities_per_district` (a pandas Series or DataFrame).

In [ ]:
# Hint: facilities.groupby('district') groups rows by district.
# .size() counts how many rows are in each group.
# .sort_values(ascending=False) sorts those counts from most to fewest.

facilities_per_district = facilities.groupby('district')['facility_id'].count().sort_values(ascending=False)

print(facilities_per_district)


---
### ✅ Knowledge Check — Counting facilities per district

*No coding here — just check your understanding before moving on.*

**Q1. District A has 2 health facilities and 200,000 people. District B has 8 facilities and 150,000 people. Which district is more likely to be underserved relative to its population?**

- **A.** District A — fewer facilities for more people
- **B.** District B — it has more facilities so it must have more demand
- **C.** Both are equally served since we don't know facility size
- **D.** Underserved status can't be judged from facility counts at all

<details>
<summary>Show answer</summary>

**Answer: A** — District A has roughly 100,000 people per facility versus District B's ~19,000 — a raw facility count only becomes meaningful once it's related to the population it has to serve.

</details>

**Q2. Why is a simple facility count per district a useful *first* step in spatial epidemiology, even though it's a crude measure?**

- **A.** It directly measures disease burden
- **B.** It's a quick way to spot possible gaps in health infrastructure before doing deeper analysis
- **C.** It replaces the need for population data entirely
- **D.** It accounts for travel time and road access automatically

<details>
<summary>Show answer</summary>

**Answer: B** — Facility counts are a cheap, fast screening tool to flag districts that deserve closer investigation — they don't account for population, distance, or facility capacity on their own.

</details>



In [ ]:
# Tests hidden — run your solution above first
pass

---
## ✅ Worked example 2 — Scatter map of facility locations

In [ ]:
# Drop facilities with missing coordinates for mapping.
mapped = facilities.dropna(subset=['latitude', 'longitude'])

type_colours = {'hospital': '#e63946', 'clinic': '#457b9d', 'health_post': '#2a9d8f'}
fig, ax = plt.subplots(figsize=(8, 6))

for ftype, colour in type_colours.items():
    sub = mapped[mapped['facility_type'] == ftype]
    ax.scatter(sub['longitude'], sub['latitude'], c=colour, s=60, label=ftype, alpha=0.8, edgecolors='white', linewidths=0.5)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Health facilities by type — synthetic data')
ax.legend(title='Facility type')
ax.set_xlim(25.0, 33.5)
ax.set_ylim(-23.0, -15.0)

# Add a real-world OpenStreetMap basemap for context. This needs internet
# access to fetch map tiles — if that fails (e.g. no connectivity at the
# venue), we fall back to the plain scatter map above instead of crashing.
try:
    import contextily as cx
    cx.add_basemap(ax, crs='EPSG:4326', source=cx.providers.CartoDB.Positron, zoom=6)
except Exception as e:
    print(f'(Basemap unavailable — showing plain scatter map instead: {e})')

plt.tight_layout()
plt.show()

print(f'Mapped {len(mapped)} of {len(facilities)} facilities ({len(facilities)-len(mapped)} missing coords skipped)')


---
## 📖 2. The silent float-corruption bug — why IDs must be strings

Imagine your facility IDs are `F001`, `F002`, …, `F060`. Now imagine someone exports the dataset as Excel, and Excel "helpfully" converts them to integers: `1`, `2`, …, `60`. When you re-import that file with `pd.read_csv()`, pandas may read those as `float64` — so `1` becomes `1.0`. Now merging on `facility_id` silently fails because `'F001' != 1.0`.

**The fix:** always explicitly tell pandas the dtype of ID columns:

In [ ]:
# Without dtype override — pandas guesses.
df_auto = pd.read_csv(DATA_DIR / 'health_facilities.csv')
print('Auto-inferred facility_id dtype:', df_auto['facility_id'].dtype)

# With explicit string dtype — safe.
df_safe = pd.read_csv(DATA_DIR / 'health_facilities.csv', dtype={'facility_id': str})
print('Explicit string facility_id dtype:', df_safe['facility_id'].dtype)
print('First three IDs:', df_safe['facility_id'].head(3).tolist())

---
## 🧪 Exercise 2 — District with the highest total catchment population  *(3 points)*

Compute the **total** `catchment_pop` for each district (include all facilities, even those with missing coordinates — they still serve a population). Store the name of the district with the highest total in a variable called `top_district` (a string).

In [ ]:
# Hint: facilities.groupby('district')['catchment_pop'].sum() gives the
# total catchment population for each district. Store that result in
# district_catchment, then use district_catchment.idxmax() to find the
# district name with the highest total — store that in top_district.

district_catchment = facilities.groupby('district')['catchment_pop'].sum()
top_district = district_catchment.idxmax()

print(f'District with highest total catchment population: {top_district}')
print(f'  Total: {district_catchment[top_district]:,}')


---
### ✅ Knowledge Check — Catchment population

*No coding here — just check your understanding before moving on.*

**Q1. Why look at *total* catchment population per district rather than just the number of facilities?**

- **A.** Because population data is easier to collect than facility counts
- **B.** Because the same number of facilities can serve very different sized populations, changing the real burden per facility
- **C.** Total population is only useful for elections, not health planning
- **D.** Catchment population and facility count always move together

<details>
<summary>Show answer</summary>

**Answer: B** — Two districts with identical facility counts can have very different real-world strain on services if one has a much larger population relying on those same facilities.

</details>

**Q2. The exercise says to include facilities even when their coordinates are missing. Why?**

- **A.** Missing coordinates means the facility doesn't really exist
- **B.** A facility still serves a real population even if we don't yet know exactly where it sits on a map
- **C.** It makes the numbers add up to a rounder total
- **D.** Facilities without coordinates are always duplicates

<details>
<summary>Show answer</summary>

**Answer: B** — Missing GPS coordinates is a data quality gap, not evidence the facility or its population doesn't exist — dropping it would understate the true catchment population.

</details>



In [ ]:
# Tests hidden — run your solution above first
pass

---
## 🏁 Recap

- Geocoded health data links every record to a coordinate in WGS 84 (lat/lon).
- Always check for missing coordinates before mapping — silently dropping them hides data gaps.
- Always load ID columns as strings to avoid the float-corruption bug.
- A scatter plot of lat/lon is the fastest way to sanity-check spatial data: outliers, clusters, and systematic errors are immediately visible.

**Next:** Notebook 02 introduces the gridded environmental datasets we will link to these facility locations.

In [ ]:
# Score summary (run at the end to see your total).
try:
    import nbgrader.apps.api as _ng
    print(_ng.NbGraderAPI().get_student_notebook_score('01_spatial_health_data'))
except Exception:
    print('Score tracking is managed by the course server. Ask your facilitator for your score.')